# Streamflow Data Cleaning: USGS Iowa Gauge Stations

Cleans the **gauge-station metadata** table for the USGS daily-discharge sites
into a tidy, type-safe table keyed on `site_no`, ready to join discharge
observations to a location and watershed.

**Input:**  `data/tabular/01_raw/streamflow/usgs-iowa-gauges.csv`
**Output:** `data/tabular/02_clean/streamflow/usgs-iowa-gauges-clean.csv`

Each row describes one USGS streamgage and its location attributes:

| raw column        | meaning                                            | units   |
|-------------------|----------------------------------------------------|---------|
| `site_no`         | USGS site number (the canonical join key)          | id      |
| `station_name`    | human-readable gage name                           | text    |
| `latitude`        | decimal latitude (NAD83)                           | degrees |
| `longitude`       | decimal longitude (NAD83)                          | degrees |
| `drain_area_sqmi` | contributing drainage area above the gage          | mi²     |
| `huc8`            | 8-digit hydrologic unit code (subbasin)            | id      |
| `county_fips`     | 3-digit county FIPS (Iowa state FIPS = 19)         | id      |

**Cleaning steps:**

1. Load the raw extract, reading every identifier column as a string so leading
   zeros survive.
2. **Normalize `station_name`** whitespace (strip, collapse runs of spaces).
3. **Repair `huc8`** — the export stripped the leading zero from Region-07 codes
   and a handful of rows carry a longer HUC10/HUC12 code; restore the leading
   zero and truncate to a clean 8-digit subbasin HUC8.
4. **Zero-pad `county_fips`** to the 3-digit FIPS county convention.
5. **Round** `latitude`/`longitude` to 6 decimals (~0.1 m) to kill export noise.
6. **Range-validate** the coordinates, drainage area, and id formats.
7. **De-duplicate** on `site_no` so the join key is unique.
8. **Sort & write** the tidy table to `02_clean`.

> **Identifier note:** `site_no` is **not** fixed-width. Surface-water gages use
> 8 digits, but a few sites here use 10- or 15-digit downstream-order / lat-long
> site numbers; all are valid USGS IDs and are kept verbatim as strings. Nulls in
> `drain_area_sqmi` are **kept**, not imputed — missingness handling is a modeling
> decision left to the pipeline downstream.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    """Walk upward until we find the repo's data/tabular directory.

    Notebooks have no __file__, and the kernel's working directory varies, so
    resolving paths relative to a fixed number of "../" is fragile. Searching
    upward for a sentinel makes the notebook runnable from anywhere.
    """
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "data" / "tabular").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing data/tabular/")


REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / "data" / "tabular" / "01_raw" / "streamflow"
CLEAN_DIR = REPO_ROOT / "data" / "tabular" / "02_clean" / "streamflow"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO_ROOT)
print("Raw dir:  ", RAW_DIR)
print("Clean dir:", CLEAN_DIR)

Repo root: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction
Raw dir:   /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/01_raw/streamflow
Clean dir: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/02_clean/streamflow


## Step 1 — Load

~700 gages. Every identifier (`site_no`, `huc8`, `county_fips`) is read as a
string so the export's leading zeros are preserved; the coordinates and drainage
area are genuine floats.

In [2]:
df = pd.read_csv(
    RAW_DIR / "usgs-iowa-gauges.csv",
    dtype={"site_no": "string", "huc8": "string", "county_fips": "string"},
)
n_raw = len(df)
print(f"Loaded {n_raw:,} rows")
print("Columns:", list(df.columns))
print("\nNulls per column:")
print(df.isna().sum().to_string())
df.head()

Loaded 703 rows
Columns: ['site_no', 'station_name', 'latitude', 'longitude', 'drain_area_sqmi', 'huc8', 'county_fips']

Nulls per column:
site_no             0
station_name        0
latitude            0
longitude           0
drain_area_sqmi    11
huc8                0
county_fips         0


,site_no,station_name,latitude,longitude,drain_area_sqmi,huc8,county_fips
0,05317650,"Blue Earth River near Lakota, IA",43.429679,-94.069958,64.6,7020009,109
1,05387300,"Upper Iowa River at Chester, IA",43.491077,-92.362116,141.0,7060002,89
2,05387400,"Upper Iowa River near Kendallville, IA",43.464611,-92.038889,273.0,7060002,191
3,05387440,"Upper Iowa River at Bluffton, IA",43.406913,-91.899046,367.0,7060002,191
4,05387500,"Upper Iowa River at Decorah, IA",43.304889,-91.795543,511.0,7060002,191


## Step 2 — Normalize station names

A few names arrive with doubled or trailing spaces (e.g. embedded site labels
like `075N44W22DAC            Missouri River at C Bluffs`). Strip the ends and
collapse internal runs of whitespace to a single space. Casing is left untouched
to preserve fidelity with the USGS source.

In [3]:
before = df["station_name"].str.contains(r"\s{2,}").sum()
df["station_name"] = (
    df["station_name"].str.replace(r"\s+", " ", regex=True).str.strip()
)
print(f"Names with collapsed whitespace: {before}")
assert not df["station_name"].str.contains(r"\s{2,}").any()
assert df["station_name"].str.len().gt(0).all(), "empty station name"

Names with collapsed whitespace: 3


## Step 3 — Repair the HUC8 subbasin code

The column is named `huc8` but the raw values are inconsistent:

* Region-07 (Upper Mississippi) codes lost their leading zero on export and
  arrive 7 digits long — `7020009` should be `07020009`.
* A handful of rows carry a finer HUC10/HUC12 code (9–12 digits) instead of the
  subbasin HUC8.

A valid HUC always has an **even** number of digits, so an odd length means a
stripped leading zero. Restore it (pad odd → even) and then truncate to the
first 8 digits to recover the subbasin HUC8 every gage should have.

In [4]:
def normalize_huc8(huc: str) -> str:
    if len(huc) % 2 == 1:        # odd length ⇒ a leading zero was stripped
        huc = "0" + huc
    return huc[:8]               # truncate HUC10/HUC12 down to the subbasin


print("Before — length distribution:")
print(df["huc8"].str.len().value_counts().sort_index().to_string())

df["huc8"] = df["huc8"].map(normalize_huc8)

print("\nAfter — length distribution:")
print(df["huc8"].str.len().value_counts().to_string())
assert df["huc8"].str.fullmatch(r"\d{8}").all(), "huc8 not 8 digits"
print(f"\nDistinct HUC8 subbasins: {df['huc8'].nunique()}")

Before — length distribution:
huc8
7     466
8     229
9       1
11      5
12      2

After — length distribution:
huc8
8    703

Distinct HUC8 subbasins: 53


## Step 4 — Zero-pad county FIPS

`county_fips` is the **county** portion of the FIPS code (Iowa's state FIPS is
`19`). It arrives as a bare integer (`1`–`197`), so zero-pad to the canonical
3-digit width. Iowa's 99 counties use the odd codes `001`–`197`.

In [5]:
print("Before — length distribution:", df["county_fips"].str.len().value_counts().to_dict())
df["county_fips"] = df["county_fips"].str.zfill(3)
print("After  — length distribution:", df["county_fips"].str.len().value_counts().to_dict())
assert df["county_fips"].str.fullmatch(r"\d{3}").all(), "county_fips not 3 digits"
codes = df["county_fips"].astype(int)
assert codes.between(1, 197).all(), "county_fips outside Iowa range"

Before — length distribution: {np.int64(3): 385, np.int64(2): 296, np.int64(1): 22}
After  — length distribution: {np.int64(3): 703}


## Step 5 — Round coordinates

USGS reports latitude/longitude at varying precision; round to 6 decimals
(~0.1 m at this latitude), which is well within gage-location accuracy and drops
spurious float-export digits.

In [6]:
df["latitude"] = df["latitude"].round(6)
df["longitude"] = df["longitude"].round(6)
print("latitude  range:", df["latitude"].min(), "→", df["latitude"].max())
print("longitude range:", df["longitude"].min(), "→", df["longitude"].max())

latitude  range: 40.393655 → 43.500247
longitude range: -96.597534 → -90.190406


## Step 6 — Validate ranges

Bound the numerics to physically valid values. Coordinates must sit within a
generous Iowa bounding box (the dataset includes border-river gages on the
Missouri and Big Sioux), and drainage area must be non-negative. `drain_area_sqmi`
nulls are ignored by the check and kept as-is.

In [7]:
assert df["latitude"].between(40.0, 43.7).all(), "latitude outside Iowa box"
assert df["longitude"].between(-97.0, -90.0).all(), "longitude outside Iowa box"
assert (df["drain_area_sqmi"].dropna() >= 0).all(), "negative drainage area"

print("drain_area_sqmi (non-null):")
print(f"  range [{df['drain_area_sqmi'].min()}, {df['drain_area_sqmi'].max()}] mi²")
print(f"  nulls kept: {df['drain_area_sqmi'].isna().sum()}")
print("\nAll range checks passed.")

drain_area_sqmi (non-null):
  range [0.02, 316200.0] mi²
  nulls kept: 11

All range checks passed.


## Step 7 — De-duplicate

`site_no` should uniquely identify a gage. Drop exact duplicate rows and confirm
the key is unique and non-null.

In [8]:
df = df.drop_duplicates()
dup_keys = df["site_no"].duplicated().sum()
print(f"Duplicate site_no values: {dup_keys}")
assert dup_keys == 0, "duplicate site_no remains"
assert df["site_no"].notna().all(), "null site_no"
print(f"Rows after de-duplication: {len(df):,} (from {n_raw:,})")

Duplicate site_no values: 0
Rows after de-duplication: 703 (from 703)


## Step 8 — Sort & write

Lead with the `site_no` key, sort by it, and write the tidy table to `02_clean`.

In [9]:
ordered = [
    "site_no",
    "station_name",
    "latitude",
    "longitude",
    "drain_area_sqmi",
    "huc8",
    "county_fips",
]
df = df[ordered].sort_values("site_no").reset_index(drop=True)

out_path = CLEAN_DIR / "usgs-iowa-gauges-clean.csv"
df.to_csv(out_path, index=False)
print(f"Wrote {len(df):,} rows × {df.shape[1]} cols to:")
print(" ", out_path.relative_to(REPO_ROOT))
df.head()

Wrote 703 rows × 7 cols to:
  data/tabular/02_clean/streamflow/usgs-iowa-gauges-clean.csv


,site_no,station_name,latitude,longitude,drain_area_sqmi,huc8,county_fips
0,05317650,"Blue Earth River near Lakota, IA",43.429680,-94.069958,64.6,07020009,109
1,05387300,"Upper Iowa River at Chester, IA",43.491077,-92.362116,141.0,07060002,089
2,05387400,"Upper Iowa River near Kendallville, IA",43.464611,-92.038889,273.0,07060002,191
3,05387440,"Upper Iowa River at Bluffton, IA",43.406913,-91.899046,367.0,07060002,191
4,05387500,"Upper Iowa River at Decorah, IA",43.304889,-91.795543,511.0,07060002,191
